In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# บันทึกไฟล์ไปที่โฟลเดอร์ paper_figures ที่อยู่ใน Work15 (ย้อนกลับไป 1 โฟลเดอร์)
OUT = Path("../paper_figures")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

METHOD_COLORS = {
    "Single Holdout": "#444444",
    "Expanding Window": "#0072B2",
    "Sliding Window": "#D55E00",
}

def save_figure(fig, filename):
    fig.savefig(
        OUT / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)

In [2]:
def make_figure_3(files, initial_valuation_date):
    """
    files:
        dict mapping method names to verified original CSV paths.

    initial_valuation_date:
        verified valuation date of the initial capital,
        immediately before the first recorded return interval.
        Do not guess this date.
    """
    if initial_valuation_date is None:
        raise ValueError(
            "Provide the verified initial valuation date first."
        )

    base_date = pd.Timestamp(initial_valuation_date)
    frames = {}

    for method, path in files.items():
        df = pd.read_csv(path)

        required = {"date", "step_return"}
        if not required.issubset(df.columns):
            raise ValueError(
                f"{method}: missing columns {required - set(df.columns)}"
            )

        df["date"] = pd.to_datetime(df["date"], errors="raise")
        df["step_return"] = pd.to_numeric(
            df["step_return"], errors="raise"
        )

        if df.empty:
            raise ValueError(f"{method}: empty file")

        if df["date"].isna().any():
            raise ValueError(f"{method}: missing date")

        if df["date"].duplicated().any():
            raise ValueError(f"{method}: duplicate dates")

        if not df["date"].is_monotonic_increasing:
            raise ValueError(f"{method}: dates are not ordered")

        returns = df["step_return"].to_numpy(dtype=float)

        if not np.isfinite(returns).all():
            raise ValueError(f"{method}: non-finite returns")

        if (returns <= -1).any():
            raise ValueError(
                f"{method}: return <= -100%; inspect the source data"
            )

        if not base_date < df["date"].iloc[0]:
            raise ValueError(
                f"{method}: initial date must precede the first return date"
            )

        frames[method] = df.reset_index(drop=True)

    methods = list(frames)
    reference_dates = frames[methods[0]]["date"]

    for method in methods[1:]:
        if not frames[method]["date"].equals(reference_dates):
            raise ValueError(
                "Return dates differ between methods. "
                "Inspect interval alignment; do not fill missing returns."
            )

    plot_dates = pd.DatetimeIndex(
        [base_date] + reference_dates.tolist()
    )

    fig, axes = plt.subplots(
        2, 1, figsize=(10, 7), sharex=True,
        gridspec_kw={"height_ratios": [2, 1]}
    )

    for method, df in frames.items():
        returns = df["step_return"].to_numpy(dtype=float)

        index_value = np.concatenate([
            [100.0],
            100.0 * np.cumprod(1.0 + returns),
        ])

        running_peak = np.maximum.accumulate(index_value)
        drawdown_pct = 100.0 * (
            index_value / running_peak - 1.0
        )

        color = METHOD_COLORS[method]

        axes[0].plot(
            plot_dates, index_value, label=method, color=color
        )
        axes[1].plot(
            plot_dates, drawdown_pct, label=method, color=color
        )

        print(
            method,
            "return observations =", len(df),
            "first return date =", df["date"].iloc[0],
            "last return date =", df["date"].iloc[-1],
        )

    axes[0].set_ylabel("Portfolio index: initial capital = 100")
    axes[1].set_ylabel("Drawdown (%)")
    axes[1].set_xlabel("Valuation date")
    axes[0].legend()

    for ax in axes:
        ax.grid(alpha=0.2)

    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].xaxis.set_major_formatter(
        mdates.DateFormatter("%Y-%m")
    )

    fig.suptitle("Common-period OOS portfolio paths")
    fig.text(
        0.01, 0.01,
        "Original-run returns; unresolved implementation issues. "
        "No missing returns were filled.",
        fontsize=9,
    )
    fig.tight_layout(rect=(0, 0.05, 1, 0.96))
    save_figure(fig, "figure_3_common_oos_paths")


# Call only after verifying the three file identities
# and the initial valuation date.
#
# files = {
#     "Single Holdout": "...verified Single Holdout CSV...",
#     "Expanding Window": "...verified Expanding Fold 5 CSV...",
#     "Sliding Window": "...verified Sliding Fold 5 CSV...",
# }
#
# make_figure_3(files, initial_valuation_date="...verified date...")


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# บันทึกไฟล์ไปที่โฟลเดอร์ paper_figures ที่อยู่ใน Work15 (ย้อนกลับไป 1 โฟลเดอร์)
OUT = Path("../paper_figures")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10.5,
    "legend.fontsize": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.0,
})

METHOD_COLORS = {
    "Single Holdout": "#444444",
    "Expanding Window": "#0072B2",
    "Sliding Window": "#D55E00",
}

def save_figure(fig, filename):
    fig.savefig(
        OUT / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
    )
    plt.close(fig)


def make_figure_3(files, initial_valuation_date):
    if initial_valuation_date is None:
        raise ValueError("Provide the verified initial valuation date first.")

    base_date = pd.Timestamp(initial_valuation_date)
    frames = {}

    for method, path in files.items():
        df = pd.read_csv(path)

        required = {"date", "step_return"}
        if not required.issubset(df.columns):
            raise ValueError(
                f"{method}: missing columns {required - set(df.columns)}"
            )

        df["date"] = pd.to_datetime(df["date"], errors="raise")
        df["step_return"] = pd.to_numeric(df["step_return"], errors="raise")

        if df.empty:
            raise ValueError(f"{method}: empty file")
        if df["date"].isna().any():
            raise ValueError(f"{method}: missing date")
        if df["date"].duplicated().any():
            raise ValueError(f"{method}: duplicate dates")
        if not df["date"].is_monotonic_increasing:
            raise ValueError(f"{method}: dates are not ordered")

        returns = df["step_return"].to_numpy(dtype=float)

        if not np.isfinite(returns).all():
            raise ValueError(f"{method}: non-finite returns")
        if (returns <= -1).any():
            raise ValueError(
                f"{method}: return <= -100%; inspect the source data"
            )
        if not base_date < df["date"].iloc[0]:
            raise ValueError(
                f"{method}: initial date must precede the first return date"
            )

        frames[method] = df.reset_index(drop=True)

    methods = list(frames)
    reference_dates = frames[methods[0]]["date"]

    for method in methods[1:]:
        if not frames[method]["date"].equals(reference_dates):
            raise ValueError(
                "Return dates differ between methods. "
                "Inspect interval alignment; do not fill missing returns."
            )

    plot_dates = pd.DatetimeIndex([base_date] + reference_dates.tolist())

    # ปรับสัดส่วนเป็นแนวตั้ง (กว้าง 5.4 นิ้ว, สูง 6.2 นิ้ว)
    fig, axes = plt.subplots(
        2, 1, figsize=(5.4, 6.2), sharex=True,
        gridspec_kw={"height_ratios": [1.8, 1.1]}
    )

    for method, df in frames.items():
        returns = df["step_return"].to_numpy(dtype=float)

        index_value = np.concatenate([
            [100.0],
            100.0 * np.cumprod(1.0 + returns),
        ])

        running_peak = np.maximum.accumulate(index_value)
        drawdown_pct = 100.0 * (index_value / running_peak - 1.0)

        color = METHOD_COLORS[method]

        axes[0].plot(
            plot_dates, index_value, label=method, color=color, linewidth=1.6
        )
        axes[1].plot(
            plot_dates, drawdown_pct, label=method, color=color, linewidth=1.4
        )

        print(
            method,
            "return observations =", len(df),
            "first return date =", df["date"].iloc[0],
            "last return date =", df["date"].iloc[-1],
        )

    axes[0].set_ylabel("Portfolio Index (Init = 100)", fontweight="bold")
    axes[1].set_ylabel("Drawdown (%)", fontweight="bold")
    axes[1].set_xlabel("Valuation Date", fontweight="bold")

    # วาง Legend ด้านบนกราฟเพื่อไม่ให้บังเส้นกราฟ
    axes[0].legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        frameon=True,
    )

    for ax in axes:
        ax.grid(alpha=0.25)

    axes[1].axhline(0, color="black", linewidth=0.9)
    
    # ตั้งค่าระยะห่างแกน X ทุกๆ 6 เดือน เพื่อไม่ให้ตัวหนังสือทับกันในคอลัมน์แคบ
    axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

    fig.tight_layout()
    save_figure(fig, "figure_3_common_oos_paths")


# รันสร้างไฟล์ตามเดิม
files = {
    "Single Holdout": "Single Holdout/oos_account.csv",
    "Expanding Window": "Expanding window/oos_account.csv",
    "Sliding Window": "Sliding Window/oos_account.csv",
}

make_figure_3(files, initial_valuation_date="2023-12-29")
print("บันทึกกราฟ Figure 3 สำเร็จที่:", (OUT / "figure_3_common_oos_paths.png").resolve())

Single Holdout return observations = 645 first return date = 2024-01-04 00:00:00 last return date = 2026-07-28 00:00:00
Expanding Window return observations = 645 first return date = 2024-01-04 00:00:00 last return date = 2026-07-28 00:00:00
Sliding Window return observations = 645 first return date = 2024-01-04 00:00:00 last return date = 2026-07-28 00:00:00
บันทึกกราฟ Figure 3 สำเร็จที่: D:\Trading Algorithm Project (Github)\Trading-Algorithm-Seminar-\Work15\paper_figures\figure_3_common_oos_paths.png
